# ch02 PLC 状态机与工序分析

**目标**：基于 PLC 状态机标签拆解分拣全流程，量化各工序时序规律

**依赖**：ch01

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path('.') / '..'))
from src.utils.data_loader import load_state_machine_data
from src.utils.output_manager import save_dataframe

import pandas as pd
import numpy as np

CHAPTER_ID = 'ch02'

## 1. 数据加载

In [ ]:
df = load_state_machine_data()
df['Label'] = df['Label'].astype(int)
print(f"数据形状: {df.shape}")
print(f"状态分布:\n{df['Label'].value_counts().sort_index()}")

## 2. 状态边界识别

In [ ]:
# 识别状态切换点
state_changes = df['Label'].diff().ne(0)
change_points = df.index[state_changes]
print(f"共识别 {len(change_points)} 个状态切换点")

## 3. 状态持续时间统计

In [ ]:
# 计算每个状态的持续时间
durations = []
for i in range(len(change_points) - 1):
    start = change_points[i]
    end = change_points[i + 1]
    state = df.loc[start, 'Label']
    duration = (end - start).total_seconds()
    durations.append({'state': state, 'duration': duration})

duration_df = pd.DataFrame(durations)
state_stats = duration_df.groupby('state')['duration'].agg(['count', 'mean', 'std', 'min', 'max'])
state_stats

## 4. 状态转移矩阵

In [ ]:
# 构建状态转移概率矩阵
transitions = pd.crosstab(df['Label'], df['Label'].shift(-1), normalize='index')
transitions

## 5. 可视化（待扩展）

In [ ]:
# TODO: 绘制带状态背景色的时序图
# from src.utils.visualizer import plot_state_timeline

## 6. 保存产物

In [ ]:
save_dataframe(state_stats.reset_index(), 'state_duration_stats.csv', CHAPTER_ID)
save_dataframe(transitions.reset_index(), 'state_transition_matrix.csv', CHAPTER_ID)